In [1]:
import numpy as np

In [23]:
def read_hex_matrix(path, signed=True):
    # 각 줄: 16비트(4hex) -> 상위바이트(열 2p), 하위바이트(열 2p+1)
    A = np.zeros((16, 16), dtype=np.int8 if signed else np.uint8)
    with open(path, 'r') as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    assert len(lines) == 128, "128줄 필요"
    for idx, hx in enumerate(lines):
        v = int(hx, 16)                     # 0..65535
        hi = (v >> 8) & 0xFF                # (r, 2p)
        lo = v & 0xFF                       # (r, 2p+1)
        r  = idx // 8
        p  = idx % 8
        if signed:
            hi = hi - 256 if hi >= 128 else hi
            lo = lo - 256 if lo >= 128 else lo
        A[r, 2*p]     = hi
        A[r, 2*p + 1] = lo
    return A

def read_hex_matrix_2(path, signed=True):
    # 각 줄: 16비트(4hex) -> 상위바이트(열 2p), 하위바이트(열 2p+1)
    A = np.zeros((16, 16), dtype=np.int8 if signed else np.uint8)
    with open(path, 'r') as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    assert len(lines) == 128, "128줄 필요"
    for idx, hx in enumerate(lines):
        v = int(hx, 16)                     # 0..65535
        hi = (v >> 8) & 0xFF                # (r, 2p)
        lo = v & 0xFF             # (row1, col)
        col  = idx // 8           # 0..15 (열)
        pair = idx % 8            # 0..7  (행 쌍 인덱스)
        row0 = 2*pair
        row1 = row0 + 1
        if signed:
            hi = hi - 256 if hi >= 128 else hi
            lo = lo - 256 if lo >= 128 else lo
        A[row0, col] = hi
        A[row1, col] = lo
    return A

In [24]:
# 파일 읽기
A = read_hex_matrix(r"amem.hex", signed=True)
B = read_hex_matrix_2(r"bmem.hex", signed=True)

# 고정소수점 곱: 소수비트 f (예: 4)
f = 4

In [28]:
# 방식1: 곱마다 쉬프트 후 누적 (하드웨어와 일치하는 경우가 많음)
C1 = np.zeros((16,16), dtype=np.int32)
for i in range(16):
    for j in range(16):
        acc = 0
        for k in range(16):
            acc += (int(A[i,k]) * int(B[k,j])) >> f  # 산술 쉬프트 효과
        C1[i,j] = acc


In [27]:
print(B[:,0])

[  21   13  -54  118 -103  -67  109 -124   72   43   22  -56   -4 -112
   96  122]


In [ ]:
A1=A/16

B1=B/16

C2=np.matmul(A1,B1)

In [ ]:
print(C1[0,:]/16)
print(C2[0,:])

[ -50.375   -62.875    33.375  -179.0625 -108.0625   67.1875  -18.8125
  -49.5      21.1875   -3.1875  -21.375  -121.0625    9.875    45.8125
   78.125    51.375 ]
[ -50.0390625   -62.421875     33.74609375 -178.54296875 -107.52734375
   67.66015625  -18.36328125  -49.14453125   21.5078125    -2.78125
  -21.         -120.58203125   10.37109375   46.24609375   78.5625
   51.7421875 ]


In [ ]:
C3=np.zeros((16,16), dtype=np.)
for i in range(16):
    for j in range(16):


[[ -806 -1006   534 -2865 -1729  1075  -301  -792   339   -51  -342 -1937
    158   733  1250   822]
 [   69  2009     7 -2310     6 -2996   382  -559  2099  1552  -257  -435
  -1009  -338  -555 -1216]
 [-3825  1851   919   721  -468   315   192 -1996 -1876   825  1993     6
     29   -27   485  2155]
 [  358   537  -155  1980  -767  2028 -1063   130   430  2911  1183   562
   -593  -341  -792  -230]
 [ -356   253   343  1202 -1607  1777  -757  -584  -513    10  1344  1806
   -870   734     8  1667]
 [ -979  -321  1533    35  1008   884  -467  -469  1138  2113   197  -302
    447   129   111  1166]
 [ 2172 -1782   610  1851  2210  -388   331 -1081   -71 -3402 -1310   943
  -1514  2868 -1256   669]
 [-1625 -1087  1497  -872 -1502  -162 -1117  -584   265 -1130  1016  -639
    424  1139 -1035   832]
 [  837 -1373  -266    34  -351  -691  1293   471  -949 -2850    37   720
    370   338   655   472]
 [-1742  1879   951 -3105  -427 -2241  1096   503  2434  1930  -482 -3478
   -977  -518  10

In [43]:
print((A[0,:]))
print(A1[0,:])

print((B[:,0]))
print(B1[:,0])

f=4
# acc=0
# for i in range(1):
#     acc+=(int(A[0,i]) * int(B[i,0])) >> f
# print(acc)

# acc_2=0
# for i in range(1):
#     acc_2+=A1[0,i]*B1[i,0]
# print(acc_2)


print((int(A[0,0])*int(B[0,0])>>f)/16)

[  55   12   11   26   37  -20  100  -16  -30  118    7   74  108   17
 -106 -110]
[ 3.4375  0.75    0.6875  1.625   2.3125 -1.25    6.25   -1.     -1.875
  7.375   0.4375  4.625   6.75    1.0625 -6.625  -6.875 ]
[  21   13  -54  118 -103  -67  109 -124   72   43   22  -56   -4 -112
   96  122]
[ 1.3125  0.8125 -3.375   7.375  -6.4375 -4.1875  6.8125 -7.75    4.5
  2.6875  1.375  -3.5    -0.25   -7.      6.      7.625 ]
4.5
